Inferencing Pipeline
Uses the following features:
RT: Loads and calculates current 24 Hour Average humidity for a location (Muttenz) from openmeteo
RT: Loads and calculates the current 3 hour change in barometric pressure for a location (Muttenz) from openmeteo
RT: calculates the features derived_hour and 
RT: derived_month from the current time
and uses random values for the other features according to the readme.
The Pipeline loads the latest weather_prediction model from hopsworks and predict a rain probability for the next two hours.

In [149]:
# Imports
import hopsworks
import os
import dotenv

dotenv.load_dotenv()

# Load Feature Group
feature_group_name = "weather_prediction"

features = ['temperature_2m', 'relative_humidity_2m', 'precipitation',
       'wind_speed_10m', 'cloud_cover', 'surface_pressure', 'dew_point_2m', 'pressure_msl']
rt_features = ['derived_hour', 'derived_month', 'relative_humidity_2m_24h_avg', 'pressure_msl_3h_pct_change']
labels = ["precipation_label"]

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()

2026-04-10 20:29:21,563 INFO: Closing external client and cleaning up certificates.
2026-04-10 20:29:21,563 INFO: Connection closed.
2026-04-10 20:29:21,564 INFO: Initializing external client
2026-04-10 20:29:21,564 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-10 20:29:22,644 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926


In [151]:
# Get the training data for later normalization. Same Codeblock as in Training Pipeline

fgs = fs.get_feature_groups()
fg_versions = [fg.version for fg in fgs if fg.name == feature_group_name]

fg = fs.get_feature_group(name=feature_group_name, version=max(fg_versions))
print(f"Using Feature Group Version: {fg.version}")

fv = fs.get_feature_view("weather_prediction", version=fg.version)

training_dataset_versions = fv.get_training_datasets()
training_dataset_max_version = max([x.version for x in training_dataset_versions])
# Documentation seems to be wrong https://docs.hopsworks.ai/3.0/user_guides/fs/feature_view/training-data/#read-training-data
X_train, X_test, y_train, y_test = fv.get_train_test_split(training_dataset_version=training_dataset_max_version)
# X_train.drop(columns=["timestamp"], axis=1, inplace=True)
# X_test.drop(columns=["timestamp"], axis=1, inplace=True)


Using Feature Group Version: 3
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.19s) 
2026-04-10 20:30:33,779 INFO: Computing insert statistics
2026-04-10 20:30:33,786 INFO: Computing insert statistics


In [162]:
# Get current past humidity and barometric pressure change from openmeteo 
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.52271,
	"longitude": 7.64511,
	"hourly": ["relative_humidity_2m", "precipitation", "pressure_msl", "surface_pressure"],
	"timezone": "Europe/Berlin",
	"past_days": 2,
	"forecast_days": 1,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_relative_humidity_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(2).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["surface_pressure"] = hourly_surface_pressure

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: 47.52000045776367°N 7.639999866485596°E
Elevation: 293.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Hourly data
                         date  relative_humidity_2m  precipitation  \
0  2026-04-08 00:00:00+00:00                  73.0            0.0   
1  2026-04-08 01:00:00+00:00                  74.0            0.0   
2  2026-04-08 02:00:00+00:00                  75.0            0.0   
3  2026-04-08 03:00:00+00:00                  77.0            0.0   
4  2026-04-08 04:00:00+00:00                  82.0            0.0   
..                       ...                   ...            ...   
67 2026-04-10 19:00:00+00:00                  70.0            0.0   
68 2026-04-10 20:00:00+00:00                  73.0            0.0   
69 2026-04-10 21:00:00+00:00                  79.0            0.0   
70 2026-04-10 22:00:00+00:00                  82.0            0.0   
71 2026-04-10 23:00:00+00:00                  86.0            0.0   

    pressu

In [163]:
# Calculate Features in the same way as in the Featuer Pipeline
feature_dataframe = hourly_dataframe.copy()
# Weather Variables could depend on the time of day
feature_dataframe["derived_hour"] = feature_dataframe["date"].apply(lambda x: x.hour)
# Weather Variables could depend on the month of the year
feature_dataframe["derived_month"] = feature_dataframe["date"].apply(lambda x: x.month)
# Rain Probability could depend on the average humidity of the last 24 hours
feature_dataframe["timestamp"] = feature_dataframe["date"] # Copy to save column
feature_dataframe.set_index('date', inplace=True) # Needed for averaging over time later
feature_dataframe['relative_humidity_2m_24h_avg'] = feature_dataframe['relative_humidity_2m'].rolling('24h').mean()
# Rain Probability could depend on if the barometric pressure is rising or falling
feature_dataframe['pressure_msl_3h_pct_change'] = feature_dataframe['pressure_msl'].pct_change(periods=3)
print(feature_dataframe.info())
print(feature_dataframe.head())
print("min_timesamp", feature_dataframe["timestamp"].min())
print("max_timestamp", feature_dataframe["timestamp"].max())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 72 entries, 2026-04-08 00:00:00+00:00 to 2026-04-10 23:00:00+00:00
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   relative_humidity_2m          72 non-null     float32            
 1   precipitation                 72 non-null     float32            
 2   pressure_msl                  72 non-null     float32            
 3   surface_pressure              72 non-null     float32            
 4   derived_hour                  72 non-null     int64              
 5   derived_month                 72 non-null     int64              
 6   timestamp                     72 non-null     datetime64[ns, UTC]
 7   relative_humidity_2m_24h_avg  72 non-null     float64            
 8   pressure_msl_3h_pct_change    69 non-null     float32            
dtypes: datetime64[ns, UTC](1), float32(5), float64(1), int64(2)
me

In [164]:
# Select the relevant real time features
# Get datetime object for current hour
from datetime import datetime, timezone
import pytz

# Get the current time in Central European Time (CET/CEST)
cet = pytz.timezone('Europe/Berlin')
current_time_cet = datetime.now(cet)

# Convert the current time to UTC for comparison
current_time_utc = current_time_cet.astimezone(pytz.UTC)

# Calculate the absolute time differences in seconds
time_diffs = pd.Series((feature_dataframe.index - current_time_utc), index=feature_dataframe.index).dt.total_seconds().abs()

# Find the index of the row with the smallest absolute time difference
closest_idx = time_diffs.idxmin()

# Select the row
value_df = feature_dataframe.loc[[closest_idx]]
print(value_df[rt_features])


                           derived_hour  derived_month  \
date                                                     
2026-04-10 19:00:00+00:00            19              4   

                           relative_humidity_2m_24h_avg  \
date                                                      
2026-04-10 19:00:00+00:00                     65.916667   

                           pressure_msl_3h_pct_change  
date                                                   
2026-04-10 19:00:00+00:00                   -0.000295  


In [166]:
# Calculate random values for the non-realtime features
import random

prediction_input = value_df.copy()
for feature in features:
    prediction_input[feature] = random.uniform(X_train[feature].min(), X_train[feature].max())

# Drop timestamp column as we dont need it anymore
prediction_input.drop(columns=["timestamp"], axis=1, inplace=True)

print(prediction_input)

                           relative_humidity_2m  precipitation  pressure_msl  \
date                                                                           
2026-04-10 19:00:00+00:00             82.485306       0.654206    1020.78241   

                           surface_pressure  derived_hour  derived_month  \
date                                                                       
2026-04-10 19:00:00+00:00        953.514893            19              4   

                           relative_humidity_2m_24h_avg  \
date                                                      
2026-04-10 19:00:00+00:00                     65.916667   

                           pressure_msl_3h_pct_change  temperature_2m  \
date                                                                    
2026-04-10 19:00:00+00:00                   -0.000295       15.159884   

                           wind_speed_10m  cloud_cover  dew_point_2m  
date                                                         

In [167]:
# Load model from Hopsworks Model Registry

# get Hopsworks Model Registry handle

# Set path to download model locally
import joblib
from pathlib import Path

# ifs are used to improve prediction time of cell from around 0.7 seconds to 0.0 seconds
if "mr" not in globals().keys():
    mr = project.get_model_registry()
    
modelpath = Path("tmp")/f"weather_prediction_model_{fg.version}_download"
# Load Model from hopsworks library
import sys 
if not Path.exists(modelpath):
    model_reference = mr.get_model(name="weather_prediction_model", version=fg.version)
    model_reference.download(modelpath)
else:
    print(f"Model already exists in {modelpath}")

if "model" not in globals().keys():
    model = joblib.load(filename=modelpath/f"weather_prediction_model_{fg.version}.joblib")
else:
    print("Mdoel already loaded")

# Ensure the columns are in the correct order
prediction_input = prediction_input[features+rt_features]

rain_probability_2h = model.predict(prediction_input)
print(rain_probability_2h)

# Define Function to predict again if wanted with optimal performance and the already loaded RT Values
def predict_again():
    for feature in features:
        prediction_input[feature] = random.uniform(X_train[feature].min(), X_train[feature].max())
        rain_probability_2h = model.predict(prediction_input)
    print(rain_probability_2h)


Model already exists in tmp/weather_prediction_model_3_download
Mdoel already loaded
[0.39246378]


In [ ]:
# Rerun the prediction 
# The values range from around 0.28 to 0.4 which is a narrow range. There was no model feature evaluation so it could be that the realtime features have the strongest impact.
predict_again()


[0.40014494]
